# Create Mixed Stoichiometry Material

Load a crystal structure and substitute a fraction of one element with another to create
mixed-composition materials (e.g., Hf₁₋ₓZrₓO₂).

<h2 style="color:green">Usage</h2>

1. Make sure to select Input Materials (in the outer runtime) before running the notebook.
1. Set mixing parameters in cell 1.1. below.
1. Click "Run" > "Run All" to run all cells.
1. Scroll down to view results.

## How it works

1. Create a supercell from the input material
2. Find all sites of the element to replace (`ORIGINAL_ELEMENT`)
3. Replace a fraction (`FRACTION`) of them with `REPLACEMENT_ELEMENT`
4. Preview and export the mixed-composition material

## 1. Prepare the Environment
### 1.1. Set mixing parameters

In [ ]:
ORIGINAL_ELEMENT = "Hf"     # Element to partially replace
REPLACEMENT_ELEMENT = "Zr"  # Element to substitute in
FRACTION = 0.25              # Fraction of ORIGINAL_ELEMENT sites to replace (0–1)

# Supercell size — larger cells allow finer composition control.
SUPERCELL_MATRIX = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]

# Seed to randomize site selection.
RANDOM_SEED = 0

### 1.2. Install Packages

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made")

### 1.3. Get input materials

In [ ]:
from mat3ra.notebooks_utils.material import get_materials

materials = get_materials(globals())

### 1.4. Create supercell

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.made.tools.helpers import create_supercell

unit_cell = materials[0]
supercell = create_supercell(unit_cell, supercell_matrix=SUPERCELL_MATRIX)

print(f"Unit cell: {unit_cell.name}")
print(f"Supercell atoms: {len(supercell.basis.coordinates.values)}")
visualize(supercell, repetitions=[1, 1, 1], rotation="0x")

## 2. Create substitutions
### 2.1. Find sites of the original element and select which to replace

In [ ]:
import random

# Find all sites of the original element
elements = supercell.basis.elements.values
coordinates = supercell.basis.coordinates.values

original_sites = [(i, coord) for i, (el, coord) in enumerate(zip(elements, coordinates))
                   if el == ORIGINAL_ELEMENT]

n_total = len(original_sites)
n_replace = round(n_total * FRACTION)

print(f"Found {n_total} {ORIGINAL_ELEMENT} sites in the supercell.")
print(f"Will replace {n_replace} with {REPLACEMENT_ELEMENT} (fraction = {FRACTION}).")
print(f"Result: {ORIGINAL_ELEMENT}_{{{n_total - n_replace}}}{REPLACEMENT_ELEMENT}_{{{n_replace}}}")

# Select sites to replace
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
sites_to_replace = random.sample(original_sites, n_replace)

print(f"\nSites selected for substitution:")
for idx, coord in sites_to_replace:
    print(f"  Site {idx}: {coord}")

### 2.2. Apply substitutions

In [ ]:
import copy

mixed_material = copy.deepcopy(supercell)
elements = mixed_material.basis.elements.values

for idx, _ in sites_to_replace:
    elements[idx] = REPLACEMENT_ELEMENT

mixed_material.basis.elements.values = elements

# Verify composition and actual fraction
from collections import Counter
composition = Counter(mixed_material.basis.elements.values)
actual_fraction = composition[REPLACEMENT_ELEMENT] / (composition.get(ORIGINAL_ELEMENT, 0) + composition[REPLACEMENT_ELEMENT])

print(f"Composition: {dict(composition)}")
print(f"Total atoms: {sum(composition.values())}")
print(f"Requested fraction: {FRACTION}")
print(f"Actual fraction:    {actual_fraction:.4f}  ({composition[REPLACEMENT_ELEMENT]}/{composition.get(ORIGINAL_ELEMENT, 0) + composition[REPLACEMENT_ELEMENT]} {ORIGINAL_ELEMENT} sites)")


## 3. Visualize result

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize([{"material": supercell, "title": f"Original ({unit_cell.name})"},
           {"material": mixed_material, "title": f"Mixed: {dict(composition)}"}],
          rotation="-90x")

## 4. Pass to outside runtime

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

comp_str = "".join(f"{el}{n}" for el, n in sorted(composition.items()))
mixed_material.name = f"{unit_cell.name} mixed {comp_str}"
set_materials([mixed_material])